In [ ]:
!pip install transformers accelerate quanto==0.0.11 bitsandbytes
!pip install auto-gptq autoawq optimum
!pip install sentencepiece protobuf

In [8]:
from copy import deepcopy
import re
import torch
import torch.nn as nn
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1) Foundations: Numeric Data Types & Precision

![](https://substackcdn.com/image/fetch/$s_!-PZj!,w_1456,c_limit,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F8ffa0c54-88bf-45c1-8636-bdb097bb8e6b_1172x848.png)

Before we can compress a model, we need to understand HOW numbers are stored.
Neural network weights are floating-point numbers. Quantization maps them to a lower-precision representation (fewer bits) to save memory and speed up inference, at the cost of some precision loss.

### Key Data Types:
- **FP32** — 32-bit float (1 sign + 8 exponent + 23 mantissa) — full precision
- **FP16** — 16-bit float (1 sign + 5 exponent + 10 mantissa) — half precision
- **BF16** — 16-bit bfloat (1 sign + 8 exponent + 7 mantissa) — same range as FP32, less precision
- **INT8** — 8-bit integer (−128 to 127 or 0 to 255)
- **INT4** — 4-bit integer (−8 to 7 or 0 to 15)

### Memory Rule of Thumb:
- A model with $N$ parameters in FP32 uses $N \times 4$ bytes.
- Casting to BF16/FP16 halves that to $N \times 2$ bytes.
- Quantizing to INT8 quarters it to $N \times 1$ byte.
- Quantizing to INT4 reduces it to $N \times 0.5$ bytes.

In [3]:
print("=" * 60)
print("SECTION 1: Numeric Data Types & Precision")
print("=" * 60)

# Understand the numeric limits of each dtype the hardware supports.
print("\n--- Data-type ranges ---")
print("uint8 : ", torch.iinfo(torch.uint8))   # 0 .. 255
print("int8  : ", torch.iinfo(torch.int8))     # -128 .. 127
print("bfloat16: ", torch.finfo(torch.bfloat16))
print("float16 : ", torch.finfo(torch.float16))
print("float32 : ", torch.finfo(torch.float32))

# Store the same irrational value (1/3) in different precisions and observe
# how many decimal digits are faithfully preserved.
value = 1 / 3
tensor_fp64 = torch.tensor(value, dtype=torch.float64)
tensor_fp32 = torch.tensor(value, dtype=torch.float32)
tensor_fp16 = torch.tensor(value, dtype=torch.float16)
tensor_bf16 = torch.tensor(value, dtype=torch.bfloat16)

# Precision of 1/3 across dtypes
print(f"fp64 : {tensor_fp64.item():.60f}")
print(f"fp32 : {tensor_fp32.item():.60f}")
print(f"fp16 : {tensor_fp16.item():.60f}")
print(f"bf16 : {tensor_bf16.item():.60f}")

# BF16 keeps the same exponent range as FP32
# it can represent very large and very small values, but introduces rounding error

# Generate random FP32 values
# (num_samples,) = (1000,)
tensor_fp32 = torch.rand(1000, dtype=torch.float32)

# Cast down to BF16 — some precision is lost
# (num_samples,) = (1000,)
tensor_bf16 = tensor_fp32.to(dtype=torch.bfloat16)

print("FP32 sample: ", tensor_fp32[:5])
print("BF16 sample: ", tensor_bf16[:5])

mean_error = torch.abs(tensor_fp32 - tensor_bf16.float()).mean().item()
max_error  = torch.abs(tensor_fp32 - tensor_bf16.float()).max().item()
print(f"Mean absolute error: {mean_error:.8f}")
print(f"Max  absolute error: {max_error:.8f}")

SECTION 1: Numeric Data Types & Precision

--- Data-type ranges ---
uint8 :  iinfo(min=0, max=255, dtype=uint8)
int8  :  iinfo(min=-128, max=127, dtype=int8)
bfloat16:  finfo(resolution=0.01, min=-3.38953e+38, max=3.38953e+38, eps=0.0078125, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=bfloat16)
float16 :  finfo(resolution=0.001, min=-65504, max=65504, eps=0.000976562, smallest_normal=6.10352e-05, tiny=6.10352e-05, dtype=float16)
float32 :  finfo(resolution=1e-06, min=-3.40282e+38, max=3.40282e+38, eps=1.19209e-07, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=float32)
fp64 : 0.333333333333333314829616256247390992939472198486328125000000
fp32 : 0.333333343267440795898437500000000000000000000000000000000000
fp16 : 0.333251953125000000000000000000000000000000000000000000000000
bf16 : 0.333984375000000000000000000000000000000000000000000000000000
FP32 sample:  tensor([0.2061, 0.2509, 0.8019, 0.7651, 0.8154])
BF16 sample:  tensor([0.2061, 0.2500, 0.8008, 0.7656, 0.8164], dty

In [4]:
# Toy model to visualise dtype casting effects
def print_param_dtype(model):
    """Print name and dtype of every parameter in a model."""
    for name, param in model.named_parameters():
        print(f"  {name:40s} → {param.dtype}")

class DummyModel(nn.Module):
    """
    A minimal transformer-like block: embedding → 2×(linear + layernorm) → head.
    Used purely to demonstrate dtype casting and quantization effects.
    """
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(2, 2)
        self.linear_1       = nn.Linear(2, 2)
        self.layernorm_1    = nn.LayerNorm(2)
        self.linear_2       = nn.Linear(2, 2)
        self.layernorm_2    = nn.LayerNorm(2)
        self.head           = nn.Linear(2, 2)

    def forward(self, x):
        # Embed token ids
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim=2)
        x = self.token_embedding(x)

        # First linear projection
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, hidden_dim=2)
        x = self.linear_1(x)
        x = self.layernorm_1(x)
        x = torch.relu(x)

        # Second linear projection
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, hidden_dim)
        x = self.linear_2(x)
        x = self.layernorm_2(x)
        x = torch.relu(x)

        # Classification / language-modelling head
        # (batch_num, seq_len, hidden_dim) → (batch_num, seq_len, vocab_size=2)
        x = self.head(x)
        return x

# Default model is FP32
model_fp32 = DummyModel()
print_param_dtype(model_fp32)

# .half() casts every parameter to FP16
model_fp16 = DummyModel().half()
print_param_dtype(model_fp16)

# FP16 requires a CUDA device for matmul; BF16 works on CPU.
dummy_input = torch.LongTensor([[1, 0], [0, 1]])

# FP32 forward pass
# (batch_num=2, seq_len=2) → (batch_num=2, seq_len=2, vocab_size=2)
logits_fp32 = model_fp32(dummy_input)

# Try FP16 on CPU — will error because FP16 matmul is GPU-only
try:
    logits_fp16 = model_fp16(dummy_input)
except Exception as error:
    print(f"\n[Expected Error] FP16 on CPU: {type(error).__name__}: {error}")

# BF16 works on CPU
model_bf16 = deepcopy(model_fp32).to(torch.bfloat16)
# (batch_num=2, seq_len=2) → (batch_num=2, seq_len=2, vocab_size=2)
logits_bf16 = model_bf16(dummy_input)

# diff
mean_diff = torch.abs(logits_bf16 - logits_fp32).mean().item()
max_diff  = torch.abs(logits_bf16 - logits_fp32).max().item()
print(f"\nFP32 vs BF16 logit difference → Mean: {mean_diff:.6f} | Max: {max_diff:.6f}")

  token_embedding.weight                   → torch.float32
  linear_1.weight                          → torch.float32
  linear_1.bias                            → torch.float32
  layernorm_1.weight                       → torch.float32
  layernorm_1.bias                         → torch.float32
  linear_2.weight                          → torch.float32
  linear_2.bias                            → torch.float32
  layernorm_2.weight                       → torch.float32
  layernorm_2.bias                         → torch.float32
  head.weight                              → torch.float32
  head.bias                                → torch.float32
  token_embedding.weight                   → torch.float16
  linear_1.weight                          → torch.float16
  linear_1.bias                            → torch.float16
  layernorm_1.weight                       → torch.float16
  layernorm_1.bias                         → torch.float16
  linear_2.weight                          → torch.float

In [6]:
# Memory Footprint Comparison

# Compare the memory used by BLIP (image-captioning model) in FP32 vs BF16.
from transformers import BlipForConditionalGeneration
model_name = "Salesforce/blip-image-captioning-base"

# Load in FP32 (default)
model_fp32_blip = BlipForConditionalGeneration.from_pretrained(model_name)
fp32_bytes = model_fp32_blip.get_memory_footprint()

# Load in BF16
model_bf16_blip = BlipForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=torch.bfloat16
)
bf16_bytes = model_bf16_blip.get_memory_footprint()

print(f"FP32 : {fp32_bytes / 1e6:.1f} MB")
print(f"BF16 : {bf16_bytes / 1e6:.1f} MB")
print(f"Ratio: {bf16_bytes / fp32_bytes * 100:.1f}%  (≈ 50% is expected)")

# Clean up GPU/CPU memory
del model_fp32_blip, model_bf16_blip

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

FP32 : 989.8 MB
BF16 : 494.9 MB
Ratio: 50.0%  (≈ 50% is expected)


# 2) Post-Training Quantization (PTQ)

## 2.1 Asymmetric (Zero-Point) Quantization

The simplest form of quantization: map a floating-point range `[min, max]` linearly to the integer range `[0, 255]` (uint8).

### Formulas:
- **scale** $= \frac{\text{max_val} - \text{min_val}}{2^{\text{bits}} - 1}$
- **zero_point** $= \text{min_val}$
- **quantize**: $q = \text{clamp}(\text{round}(\frac{x - \text{zero_point}}{\text{scale}}), 0, 2^{\text{bits}}-1)$
- **dequantize**: $\hat{x} = q \times \text{scale} + \text{zero_point}$

It is called "Asymmetric" because the zero of the float range does NOT necessarily map to the zero of the int range.

In [10]:
# Load GPT-2 as the model to quantize
model_name = "openai-community/gpt2"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token (GPT-2 doesn't define one by default)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

print(f"GPT-2 FP32 memory: {model.get_memory_footprint() / 1e6:.1f} MB")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 FP32 memory: 510.3 MB


In [11]:
# === Quantize a Single Tensor ===
def quantize_asymmetric(t, bits=8):
    """
    Asymmetric (zero-point) quantization of a float tensor to unsigned int.

    Args:
        t:    torch.Tensor — the floating-point tensor to quantize
        bits: int — target bit-width (default 8 → uint8, range 0..255)

    Returns:
        t_quant: torch.Tensor (uint8) — quantized tensor
        state:   tuple(scale, zero_point) — needed for dequantization
    """
    qmin, qmax = 0, 2**bits - 1

    min_val, max_val = t.min(), t.max()

    # Scale maps the float range to the int range
    scale = (max_val - min_val) / (qmax - qmin)

    # Zero-point: the float value that maps to integer 0
    zero_point = min_val

    # Quantize: shift, scale, round, clamp
    t_quant = (t - zero_point) / scale
    t_quant = torch.clamp(torch.round(t_quant), qmin, qmax)
    t_quant = t_quant.to(torch.uint8)

    state = (scale, zero_point)
    return t_quant, state

def dequantize_asymmetric(t_quant, state):
    """
    Reverse the asymmetric quantization.

    Args:
        t_quant: torch.Tensor (uint8)
        state:   tuple(scale, zero_point)

    Returns:
        torch.Tensor (float32) — approximate reconstruction of the original
    """
    scale, zero_point = state
    return t_quant.to(torch.float32) * scale + zero_point

# Grab one weight tensor from GPT-2's first attention block
# (model_dim, 3 * model_dim) = (768, 2304) for GPT-2
t_original = model.transformer.h[0].attn.c_attn.weight.data
print(f"\nOriginal weight shape : {t_original.shape}")
print(f"Original dtype       : {t_original.dtype}")
print(f"Value range          : [{t_original.min():.6f}, {t_original.max():.6f}]")

# Quantize
t_quant, state = quantize_asymmetric(t_original)
print(f"\nQuantized dtype       : {t_quant.dtype}")
print(f"Quantized range      : [{t_quant.min()}, {t_quant.max()}]")
print(f"Scale                : {state[0]:.8f}")
print(f"Zero-point           : {state[1]:.8f}")

# Dequantize and measure reconstruction error
t_recon = dequantize_asymmetric(t_quant, state)
abs_error = torch.abs(t_original - t_recon)
print(f"\nReconstruction error → Mean: {abs_error.mean():.8f} | Max: {abs_error.max():.8f}")


Original weight shape : torch.Size([768, 2304])
Original dtype       : torch.float32
Value range          : [-2.843634, 2.795630]

Quantized dtype       : torch.uint8
Quantized range      : [0, 255]
Scale                : 0.02211476
Zero-point           : -2.84363437

Reconstruction error → Mean: 0.00553088 | Max: 0.01105749


In [12]:
# === Quantize the Entire Model ===
def quantize_model_asymmetric(model, bits=8):
    """Quantize every parameter in the model using asymmetric quantization."""
    states = {}
    for name, param in model.named_parameters():
        param.requires_grad = False
        param.data, state = quantize_asymmetric(param.data, bits)
        states[name] = state
    return model, states

def dequantize_model(model, states):
    """Dequantize every parameter back to float32."""
    for name, param in model.named_parameters():
        param.data = dequantize_asymmetric(param.data, states[name])
    return model

# Fix dtype property so HF reports correct memory even after param dtype changes
from transformers.models.gpt2.modeling_gpt2 import GPT2Model
GPT2Model.dtype = property(lambda self: torch.float32)

quant_model, states = quantize_model_asymmetric(model)
print(f"\nQuantized GPT-2 memory : {quant_model.get_memory_footprint() / 1e6:.1f} MB")

dequant_model = dequantize_model(quant_model, states)
print(f"Dequantized GPT-2 memory: {dequant_model.get_memory_footprint() / 1e6:.1f} MB")

del model, quant_model, dequant_model  # free memory


Quantized GPT-2 memory : 137.0 MB
Dequantized GPT-2 memory: 510.3 MB


## 2.2 Symmetric (Absmax) Quantization

Symmetric quantization maps the float range `[−|max|, +|max|]` symmetrically to `[−127, +127]` (int8).

### Formula:
- **scale** $= \frac{\max(|t|)}{127}$
- **quantize**: $q = \text{clamp}(\text{round}(\frac{t}{\text{scale}}), −127, 127)$
- **dequantize**: $\hat{x} = q \times \text{scale}$


> **Pros:** Simpler (no zero-point), plays well with symmetric weight distributions.

> **Cons:** Wastes range if the distribution is heavily skewed.

In [13]:
def quantize_symmetric(t, bits=8):
    """
    Symmetric (absmax) quantization to signed int8.

    Args:
        t:    torch.Tensor (float)
        bits: int — target bit-width

    Returns:
        t_quant: torch.Tensor (int8)
        scale:   float — single scale factor for dequantization
    """
    qmax = 2**(bits - 1) - 1  # 127 for 8-bit

    # Scale is determined by the absolute maximum value
    scale = t.abs().max() / qmax

    t_quant = torch.clamp(torch.round(t / scale), -qmax, qmax)
    t_quant = t_quant.to(torch.int8)
    return t_quant, scale

def dequantize_symmetric(t_quant, scale):
    """Reverse symmetric quantization."""
    return t_quant.to(torch.float32) * scale

# Demonstrate on a small random tensor
t_demo = torch.randn(4, 4)
print(f"Original:\n{t_demo}")

t_q, s = quantize_symmetric(t_demo)
print(f"\nQuantized (int8):\n{t_q}")
print(f"Scale: {s:.6f}")

t_r = dequantize_symmetric(t_q, s)
print(f"\nReconstructed:\n{t_r}")
print(f"Max error: {(t_demo - t_r).abs().max():.6f}")

Original:
tensor([[-1.8170, -0.0835, -0.3504, -0.3384],
        [ 0.3678, -0.3918, -0.4701, -1.0247],
        [-1.2519, -0.0898,  0.9700, -0.1127],
        [-0.2655,  0.7452, -0.1736, -1.3228]])

Quantized (int8):
tensor([[-127,   -6,  -24,  -24],
        [  26,  -27,  -33,  -72],
        [ -88,   -6,   68,   -8],
        [ -19,   52,  -12,  -92]], dtype=torch.int8)
Scale: 0.014307

Reconstructed:
tensor([[-1.8170, -0.0858, -0.3434, -0.3434],
        [ 0.3720, -0.3863, -0.4721, -1.0301],
        [-1.2591, -0.0858,  0.9729, -0.1145],
        [-0.2718,  0.7440, -0.1717, -1.3163]])
Max error: 0.007137


## 2.3 Linear Quantization with Quanto (Weight-Only INT8)

`quanto` (by HuggingFace) provides linear quantization that replaces `nn.Linear` with `QLinear`. During inference, weights are stored as int8 and dequantized on-the-fly to float16/bf16 for the actual matmul.

### Key Concept: "Weight-Only" Quantization
Activations stay in float. The `QLinear` layer stores:
- quantized weight (int8)
- per-channel or per-tensor scale factor

After `quantize()` + `freeze()`, the model is ready for memory-efficient inference.

In [17]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from quanto import quantize as quanto_quantize, freeze
from accelerate.utils import named_module_tensors

def dtype_byte_size(dtype):
    """Return the byte size of a single element of the given dtype."""
    if dtype == torch.bool:
        return 1 / 8
    bit_search = re.search(r"[^\d](\d+)$", str(dtype))
    if bit_search is None:
        raise ValueError(f"Cannot infer bit-width from dtype: {dtype}")
    return int(bit_search.group(1)) // 8

def compute_module_sizes(model):
    """Compute the total byte size of each submodule (including children)."""
    module_sizes = defaultdict(int)
    for name, tensor in named_module_tensors(model, recurse=True):
        size = tensor.numel() * dtype_byte_size(tensor.dtype)
        name_parts = name.split(".")
        for idx in range(len(name_parts) + 1):
            module_sizes[".".join(name_parts[:idx])] += size
    return module_sizes

# Load Flan-T5 Small (≈ 80M params)
model_name = "google/flan-t5-small"
tokenizer_t5 = T5Tokenizer.from_pretrained(model_name)
model_t5     = T5ForConditionalGeneration.from_pretrained(model_name)

# Inference BEFORE quantization
input_text = "Hello, my name is "
input_ids  = tokenizer_t5(input_text, return_tensors="pt").input_ids
# (batch_num=1, seq_len)

outputs = model_t5.generate(input_ids, max_new_tokens=20)
# (batch_num=1, output_len)
print(f"Before quantization: '{tokenizer_t5.decode(outputs[0], skip_special_tokens=True)}'")

# Measure size before quantization
sizes_before = compute_module_sizes(model_t5)
print(f"Model size (FP32): {sizes_before[''] / 1e9:.4f} GB")

# Inspect the first encoder block BEFORE quantization
# Note: nn.Linear layers use standard float32 weights
print("\nEncoder block[0] BEFORE quantization:")
print(model_t5.encoder.block[0].layer[0])

# --- Apply INT8 weight-only quantization ---
# quantize() replaces nn.Linear → QLinear (storing int8 weights + scale)
# freeze() finalises the quantized weights (makes them non-trainable)
quanto_quantize(model_t5, weights=torch.int8, activations=None)
freeze(model_t5)

# Measure size after quantization
print(f"=== \n\n After Quantization ===")
sizes_after = compute_module_sizes(model_t5)
print(f"\nModel size (INT8): {sizes_after[''] / 1e9:.4f} GB")

# Inspect the first encoder block AFTER quantization
# Note: Linear layers are now QLinear — the weight tensor stores int8 values
# plus a float scale. You can inspect the quantized weight like so:
#   model_t5.encoder.block[0].layer[0].SelfAttention.q.weight
print("\nEncoder block[0] AFTER quantization:")
print(model_t5.encoder.block[0].layer[0])

# Inference AFTER quantization — should produce similar output
outputs_q = model_t5.generate(input_ids, max_new_tokens=20)
print(f"\nAfter quantization : '{tokenizer_t5.decode(outputs_q[0], skip_special_tokens=True)}'")

del model_t5  # free memory

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Before quantization: 'annie scott'
Model size (FP32): 0.3078 GB

Encoder block[0] BEFORE quantization:
T5LayerSelfAttention(
  (SelfAttention): T5Attention(
    (q): Linear(in_features=512, out_features=384, bias=False)
    (k): Linear(in_features=512, out_features=384, bias=False)
    (v): Linear(in_features=512, out_features=384, bias=False)
    (o): Linear(in_features=384, out_features=512, bias=False)
    (relative_attention_bias): Embedding(32, 6)
  )
  (layer_norm): T5LayerNorm()
  (dropout): Dropout(p=0.1, inplace=False)
)
=== 

 After Quantization ===

Model size (INT8): 0.3078 GB

Encoder block[0] AFTER quantization:
T5LayerSelfAttention(
  (SelfAttention): T5Attention(
    (q): QLinear(in_features=512, out_features=384, bias=False)
    (k): QLinear(in_features=512, out_features=384, bias=False)
    (v): QLinear(in_features=512, out_features=384, bias=False)
    (o): QLinear(in_features=384, out_features=512, bias=False)
    (relative_attention_bias): Embedding(32, 6)
  )
  (l

## 2.4 GPTQ: Generative Pre-Trained Transformer Quantization

### Key Ideas:
1. Based on Optimal Brain Quantization (OBQ), which uses second-order (Hessian) information to quantize weights one-at-a-time while compensating the remaining weights.
2. Processes weights column-by-column within each layer, updating the unquantized weights to compensate for the error introduced.
3. Uses a small calibration dataset (128–256 samples) to compute the Hessian ($H = 2 X^T X$ from layer inputs).
4. Achieves 4-bit weight quantization with minimal perplexity degradation.

### Strengths vs Weaknesses:
- **Strengths:** First method to achieve 4-bit LLM quantization at scale. Excellent accuracy retention. Well-supported (HF, vLLM, TensorRT-LLM).
- **Weaknesses:** Slow quantization (hours for 70B models). Accuracy depends on calibration data. Weight-only (activations remain in FP16). GPU-centric.


```python
# NOTE: Running GPTQ quantization requires a GPU and the
# `auto-gptq` library. Below we show how to LOAD a pre-quantized GPTQ model.
# To quantize your own model, you would use:
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
quantize_config = BaseQuantizeConfig(bits=4, group_size=128, damp_percent=0.1)
model = AutoGPTQForCausalLM.from_pretrained(model_name, quantize_config)
model.quantize(calibration_dataset)
model.save_quantized("./my-model-gptq")
```

```python
# Example: Loading a pre-quantized GPTQ model from HuggingFace
# (Requires: pip install auto-gptq optimum)
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "TheBloke/Llama-2-7B-GPTQ"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",        # auto-distribute across available GPUs
    trust_remote_code=False,
    revision="main"           # main branch = 4-bit, 128 group size
)

# The model is now loaded with 4-bit GPTQ weights.
# Inference works exactly the same as a normal model:
inputs = tokenizer("The capital of France is", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## 2.5 AWQ: Activation-Aware Weight Quantization

### Key Insight:
Not all weights are equally important. A small fraction of weights correspond to large activation magnitudes ("salient channels"). Keeping those salient weights at higher precision dramatically reduces quantization error.

### How it works:
1. Run a small calibration set through the model and observe activation magnitudes per channel.
2. Identify salient weight channels (those producing large activations).
3. Apply per-channel scaling: multiply salient weights by a scale factor BEFORE quantization, then divide after dequantization.
4. Optimize the scale factors to minimize total quantization error.

```python
# --- To quantize your own model with AWQ: ---
from awq import AutoAWQForCausalLM
model = AutoAWQForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4}
model.quantize(tokenizer, quant_config=quant_config)
model.save_quantized("./llama2-7b-awq")
```


```python
# Example: Loading a pre-quantized AWQ model
# (Requires: pip install autoawq)

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
model_id = "TheBloke/Llama-2-7B-AWQ"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoAWQForCausalLM.from_quantized(
    model_id,
    fuse_layers=True,   # fuse QKV projections for speed
    device_map="auto"
```

## 2.6 GGUF: CPU / Hybrid Quantization via llama.cpp

GGUF is a **file format** (not a quantization algorithm) designed for efficient storage and loading of quantized models, originally from the llama.cpp project.

### K-quants (Mixed Precision):
K-quants (introduced by "ikawrakow" in llama.cpp) use mixed precision: different layers get different bit-widths based on their importance. Examples include `Q4_K_M`, `Q5_K_M`, etc.

### Strengths:
- Runs on CPU (no GPU required!) — ideal for consumer hardware.
- Supports hybrid CPU+GPU inference (offload $N$ layers to GPU).
- Excellent ecosystem: llama.cpp, Ollama, LM Studio, GPT4All.

### Convert & quantize your own model to GGUF
- 1. Clone llama.cpp:  git clone https://github.com/ggerganov/llama.cpp
- 2. Build:            cd llama.cpp && make
- 3. Convert:          python convert_hf_to_gguf.py --input /path/to/model --output model.gguf
- 4. Quantize:         ./llama-quantize model.gguf model-Q4_K_M.gguf Q4_K_M


```python
# GGUF models are typically used via llama.cpp or its Python bindings.
# (Requires: pip install llama-cpp-python)

# --- Inference with llama-cpp-python ---
from llama_cpp import Llama
llm = Llama(
    model_path="./llama-2-7b.Q4_K_M.gguf",
    n_ctx=2048,        # context window length
    n_threads=8,       # CPU threads for computation
    n_gpu_layers=35,   # number of layers offloaded to GPU (0 = pure CPU)
    n_batch=512,       # batch size for prompt processing
)
output = llm(
    "Explain the theory of relativity simply:",
    max_tokens=200,
    temperature=0.7,
)
print(output["choices"][0]["text"])
```

## 2.7 BitsAndBytes: NF4 / INT8 Mixed-Precision

Library by Tim Dettmers (UW). Two key features:

### 8-bit (LLM.int8()):
- Decomposes matrix multiplication into two parts:
  - (a) Normal INT8 matmul for the majority of features
  - (b) FP16 matmul for "outlier" features (those with magnitude > 6.0)
- This handles the activation outlier problem that ruins naive INT8.

### 4-bit (QLoRA / NF4):
- **NF4** = "NormalFloat4" — a 4-bit data type whose quantization levels are optimally spaced for normally-distributed weights (information-theoretically optimal for Gaussian data).
- **Double quantization:** quantize the quantization constants themselves to save even more memory.
- Primarily designed for QLoRA (4-bit base model + LoRA adapters in FP16).


```python
# (Requires: pip install bitsandbytes)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- 8-bit loading (LLM.int8()) ---
model_8bit = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    load_in_8bit=True,
    device_map="auto"
)

# --- 4-bit loading (NF4, for QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",         # NormalFloat4 (optimal for Gaussian weights)
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in BF16 during forward pass
    bnb_4bit_use_double_quant=True,     # quantize the quantization constants too
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    quantization_config=bnb_config,
    device_map="auto"
)

# Inference is identical to normal models
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
inputs = tokenizer("The meaning of life is", return_tensors="pt").to(model_4bit.device)
outputs = model_4bit.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## 2.8 SmoothQuant: W8A8 (Weight AND Activation Quantization)

SmoothQuant is typically applied through inference engines:
- NVIDIA TensorRT-LLM (native support)
- MIT-HAN-LAB/smoothquant (reference implementation)

### The Problem
Activation tensors have large **OUTLIERS** (some channels have values 10–100× larger than others). This makes direct INT8 quantization of activations very lossy. However, weights are relatively smooth and easy to quantize.

### Key Insight
"Migrate difficulty from activations to weights" by inserting a per-channel smoothing factor $s_j$ that divides activations and multiplies weights BEFORE quantization:

$$Y = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W)$$

Choose $s_j = \max(|X_j|)^\alpha / \max(|W_j|)^{1-\alpha}$, with $\alpha \in [0.5, 0.75]$.

### Result
Both weights AND activations can be quantized to INT8 (W8A8), enabling the use of efficient INT8 GEMM kernels on GPUs and achieving real hardware speedups.

```python
# Conceptual Python pseudocode:
import torch
def smooth_and_quantize(X, W, alpha=0.5):
    '''
    X: activation tensor — shape (batch_num, seq_len, model_dim)
    W: weight matrix     — shape (model_dim, output_dim)
    alpha: migration strength (0 = all in weights, 1 = all in activations)
    '''
    # Compute per-channel smoothing factors
    # (model_dim,)
    act_scales = X.abs().max(dim=0).values   # max activation per channel
    wt_scales  = W.abs().max(dim=0).values   # max weight per channel

    # Smoothing factor: balance outlier magnitude between acts and weights
    # (model_dim,)
    s = (act_scales ** alpha) / (wt_scales ** (1 - alpha))

    # Apply smoothing
    # (batch_num, seq_len, model_dim) — outliers reduced
    X_smooth = X / s.unsqueeze(0).unsqueeze(0)

    # (model_dim, output_dim) — absorbs the outlier magnitude
    W_smooth = W * s.unsqueeze(1)

    # Now both X_smooth and W_smooth are "easy" to quantize to INT8
    X_int8 = quantize_symmetric(X_smooth, bits=8)
    W_int8 = quantize_symmetric(W_smooth, bits=8)

    return X_int8, W_int8
```

# 3) Quantization-Aware Training (QAT)

All methods above are **Post-Training Quantization (PTQ)**: quantize after training is complete.

**QAT** integrates quantization INTO the training loop.

### How QAT works:
1. Insert "fake quantization" nodes after weights and/or activations.
2. During **FORWARD** pass: quantize → dequantize (simulates quantization noise).
3. During **BACKWARD** pass: use the "Straight-Through Estimator" (STE) to pass gradients through the non-differentiable rounding operation.
4. The model learns to be robust to quantization noise during training.

**Advantages:** Much better accuracy at very low bit-widths (2-bit, 1-bit).
**Disadvantages:** Requires full training (or at least fine-tuning) — expensive.

In [21]:
class FakeQuantize(torch.autograd.Function):
    """
    Fake quantization: quantize → dequantize in the forward pass,
    but pass gradients straight through in the backward pass (STE).

    This simulates quantization noise during training so the model
    learns weights that are robust to rounding.
    """

    @staticmethod
    def forward(ctx, x, num_bits=8):
        qmin, qmax = -(2**(num_bits - 1)), 2**(num_bits - 1) - 1
        scale = x.abs().max() / qmax if x.abs().max() > 0 else torch.tensor(1.0)

        # Quantize (round) then dequantize
        x_quant = torch.clamp(torch.round(x / scale), qmin, qmax)
        x_dequant = x_quant * scale
        return x_dequant

    @staticmethod
    def backward(ctx, grad_output):
        # Straight-Through Estimator: pass gradient unchanged
        return grad_output, None

fake_quantize = FakeQuantize.apply
class QATLinear(nn.Module):
    """
    A linear layer with fake quantization applied to weights during training.
    During the forward pass, weights are fake-quantized (quantize then immediately
    dequantize) so the model sees the quantization noise and adapts.

    At deployment time, you replace this with actual INT8 weights.
    """
    def __init__(self, in_features, out_features, num_bits=8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.num_bits = num_bits

    def forward(self, x):
        # Fake-quantize the weights (simulate INT8 rounding noise)
        # (output_dim, input_dim) — same as self.linear.weight
        w_quant = fake_quantize(self.linear.weight, self.num_bits)

        # Standard linear operation with fake-quantized weights
        # (batch_num, *, in_features) → (batch_num, *, out_features)
        return nn.functional.linear(x, w_quant, self.linear.bias)

# Demo: compare outputs before/after fake quantization
print("\n--- QAT: Fake quantization demo ---")
torch.manual_seed(42)
layer_normal = nn.Linear(4, 4)
layer_qat    = QATLinear(4, 4, num_bits=8)
# Copy same weights so we can compare
layer_qat.linear.weight = layer_normal.weight
layer_qat.linear.bias   = layer_normal.bias

# (batch_num=2, in_features=4)
x = torch.randn(2, 4)

out_normal = layer_normal(x)
out_qat    = layer_qat(x)

print(f"FP32 output:\n{out_normal}")
print(f"\nQAT output (fake-quantized weights):\n{out_qat}")
print(f"\nDifference: {(out_normal - out_qat).abs().max():.8f}")


--- QAT: Fake quantization demo ---
FP32 output:
tensor([[ 1.5908,  0.3082,  0.0429,  0.5384],
        [ 0.3457, -0.1485,  0.0749,  0.3781]], grad_fn=<AddmmBackward0>)

QAT output (fake-quantized weights):
tensor([[ 1.5925,  0.3090,  0.0452,  0.5366],
        [ 0.3468, -0.1474,  0.0747,  0.3791]], grad_fn=<AddmmBackward0>)

Difference: 0.00222531


## 3.1 Frontier: BitNet b1.58 — Ternary / 1-bit LLMs

**Papers:**
- Wang et al. (2023): "BitNet: Scaling 1-bit Transformers for LLMs"
- Ma et al. (2024): "BitNet b1.58: The Era of 1-bit LLMs"

The idea: instead of quantizing AFTER training, train the model FROM SCRATCH with extreme low-bit weights — specifically ternary weights {-1, 0, +1}.

### Why "1.58 bits"?
With 3 possible values, information content = $\log_2(3) \approx 1.58$ bits/weight.

### BitLinear Layer:
1. **Weight quantization**: $w_q = \text{RoundClip}(w / (|w|_{\text{mean}} + \epsilon), -1, 1)$
2. **Activation quantization**: $x_q = \text{Clip}(x \times Q_b / |x|_\infty, -Q_b, Q_b)$
3. **Matrix multiply**: uses only addition/subtraction (no FP multiply needed!)
4. **Output rescaling**: $y = y_q \times \beta \times (|x|_\infty / Q_b)$

This is a **FRONTIER** research direction — the model must be trained from scratch with `BitLinear` layers. You cannot simply convert an existing model.

In [23]:
class BitLinear(nn.Module):
    """
    Weights are constrained to ternary values {-1, 0, +1} during forward pass.
    Activations are quantized per-token to a fixed range.

    True BitNet trains from scratch with
    this layer replacing all nn.Linear in the transformer. Efficient inference
    requires custom low-bit kernels (e.g., bitnet.cpp).
    """

    def __init__(self, in_features, out_features, activation_bits=8):
        super().__init__()
        # Full-precision "latent" weights (updated by optimizer)
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias   = None
        self.eps    = 1e-5
        self.Qb     = 2**(activation_bits - 1)  # e.g. 128 for 8-bit

    def ternary_quantize_weight(self, w):
        """
        Quantize weight to {-1, 0, +1} using absmean scaling.
        w_q = RoundClip(w / γ, -1, 1), where γ = mean(|w|)
        """
        gamma = w.abs().mean() + self.eps
        w_scaled = w / gamma
        # Round to nearest integer and clamp to {-1, 0, 1}
        # (output_dim, input_dim)
        w_q = torch.clamp(torch.round(w_scaled), -1, 1)
        return w_q, gamma

    def quantize_activation(self, x):
        """
        Per-token absmax activation quantization.
        x_q = Clip(x × Qb / |x|_∞, -Qb, Qb)
        """
        # Per-token absolute max
        # (batch_num, seq_len, 1)
        x_absmax = x.abs().max(dim=-1, keepdim=True).values + self.eps
        x_scale = self.Qb / x_absmax

        # Scale and clip
        # (batch_num, seq_len, model_dim)
        x_q = torch.clamp(x * x_scale, -self.Qb, self.Qb)
        return x_q, x_scale

    def forward(self, x):
        # Quantize weights to ternary {-1, 0, +1}
        w_q, gamma = self.ternary_quantize_weight(self.weight)

        # Quantize activations
        x_q, x_scale = self.quantize_activation(x)

        # Matrix multiply (with STE: pretend w_q gradients flow through)
        # (batch_num, seq_len, in_features) × (in_features, out_features) → (batch_num, seq_len, out_features)
        w_effective = w_q.detach() + self.weight - self.weight.detach()  # STE
        y = nn.functional.linear(x_q, w_effective)

        # Rescale output: undo activation scaling and apply weight scale
        # (batch_num, seq_len, out_features)
        y = y * gamma / x_scale
        return y

# Demo: BitLinear forward pass
print("--- BitLinear demo ---")
torch.manual_seed(0)
bit_layer = BitLinear(in_features=8, out_features=4)

# Inspect ternary weights
w_q, gamma = bit_layer.ternary_quantize_weight(bit_layer.weight)
print(f"Latent weight (FP32):\n{bit_layer.weight.data[:2, :4]}")
print(f"Ternary weight:\n{w_q[:2, :4]}")
print(f"Unique values: {w_q.unique().tolist()}")
print(f"Gamma (absmean scale): {gamma:.6f}")

# Forward pass
# (batch_num=2, seq_len=3, model_dim=8)
x = torch.randn(2, 3, 8)
y = bit_layer(x)
print(f"\nInput shape : {x.shape}")
print(f"Output shape: {y.shape}")
print(f"Output:\n{y[0]}")

--- BitLinear demo ---
Latent weight (FP32):
tensor([[-0.0225, -0.0230, -0.0050, -0.0087],
        [ 0.0064, -0.0253,  0.0070,  0.0062]])
Ternary weight:
tensor([[-1., -1., -0., -1.],
        [ 0., -1.,  0.,  0.]], grad_fn=<SliceBackward0>)
Unique values: [-1.0, 0.0, 1.0]
Gamma (absmean scale): 0.016957

Input shape : torch.Size([2, 3, 8])
Output shape: torch.Size([2, 3, 4])
Output:
tensor([[ 0.0075,  0.0122,  0.0188,  0.0018],
        [ 0.0622,  0.0007, -0.0728,  0.0347],
        [-0.0103, -0.0122,  0.0136, -0.0208]], grad_fn=<SelectBackward0>)


In [ ]:
'''
╔══════════════════╦══════╦═══════════╦════════════╦══════════════════════════╗
║ Method           ║ Bits ║ Type      ║ Target HW  ║ Notes                    ║
╠══════════════════╬══════╬═══════════╬════════════╬══════════════════════════╣
║ Dtype casting    ║  16  ║ N/A       ║ GPU        ║ FP32→BF16/FP16, no loss  ║
║ Zero-Point       ║  8   ║ PTQ       ║ Any        ║ Simple, per-tensor scale ║
║ Absmax           ║  8   ║ PTQ       ║ Any        ║ Symmetric, no zero-point ║
║ Quanto           ║  8   ║ PTQ       ║ GPU/CPU    ║ HF integration           ║
║ GPTQ             ║  4   ║ PTQ       ║ GPU        ║ Hessian-guided, accurate ║
║ AWQ              ║  4   ║ PTQ       ║ GPU        ║ Activation-aware scaling ║
║ GGUF (K-quants)  ║ 2-8  ║ PTQ       ║ CPU+GPU    ║ Mixed-precision, flexible║
║ BitsAndBytes     ║ 4/8  ║ PTQ       ║ GPU        ║ NF4 optimal for QLoRA    ║
║ SmoothQuant      ║  8   ║ PTQ W8A8  ║ GPU        ║ Quantizes activations too║
║ QAT (general)    ║ 4-8  ║ QAT       ║ Any        ║ Train-time, best quality ║
║ BitNet b1.58     ║ 1.58 ║ QAT/train ║ Custom     ║ Ternary {-1,0,+1}, SOTA  ║
╚══════════════════╩══════╩═══════════╩════════════╩══════════════════════════╝

Quick Decision Guide:
  - Need to run on CPU/laptop?      → GGUF (Q4_K_M)
  - GPU inference, max speed?        → AWQ + Marlin kernel
  - GPU inference, max quality?      → GPTQ or BitsAndBytes (NF4)
  - Fine-tuning on limited GPU?     → BitsAndBytes NF4 + QLoRA
  - Both weights AND activations?   → SmoothQuant (W8A8)
  - Research / next frontier?       → BitNet b1.58 (train from scratch)
'''